# HW4 — Sequence Modeling: Financial Forecasting
**CS515 Deep Learning | Sabanci University**

Bu notebook:
1. Repoyu çeker ve uv ile eksik bağımlılıkları kurar
2. Part b (exact return), Part c (rolling return), Part d (turning point) deneylerini GPU üzerinde çalıştırır
3. Sonuçları gösterir ve indirmeye hazırlar

> **Önemli:** Runtime → Change runtime type → **T4 GPU** seçili olmalı.

## 0. GPU Kontrolü

In [ ]:
import torch

print(f'CUDA available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU            : {torch.cuda.get_device_name(0)}')
    print(f'VRAM           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: GPU yok — Runtime menüsünden T4 GPU seç!')

## 1. Repo Kurulumu

In [ ]:
import os

REPO_URL = 'https://github.com/caltinuzengi/cs515-deep-learning.git'
REPO_DIR = 'cs515-deep-learning'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
else:
    print('Repo zaten mevcut, güncelleniyor...')
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print(f'Çalışma dizini: {os.getcwd()}')

## 2. Bağımlılık Kurulumu (uv)

Colab'ın CUDA-enabled torch'u korunur; sadece eksik paketler eklenir.

In [ ]:
# uv'yi kur (PyPI üzerinden)
!pip install uv -q

# Eksik paketleri sistem Python'una ekle (Colab'ın torch/CUDA'sını değiştirmez)
!uv pip install --system yfinance scikit-learn ptflops -q

# Doğrula
import yfinance, sklearn
print(f'yfinance   : {yfinance.__version__}')
print(f'scikit-learn: {sklearn.__version__}')
print(f'torch      : {torch.__version__}')

## 3. Veri İndirme ve Doğrulama

In [ ]:
!python hw4/data_pipeline.py

## 4. Part b — Exact Return Forecasting (StockLSTM & StockGRU)

In [ ]:
!python hw4/experiment_b.py --retrain

## 5. Part c — Rolling Average Return Forecasting

In [ ]:
!python hw4/experiment_c.py --retrain

## 6. Part d — Turning Point Detection (BiStockLSTM & BiStockGRU)

In [ ]:
!python hw4/experiment_d.py --retrain

## 7. Sonuçlar — Metrikler

In [ ]:
import json, glob

for path in sorted(glob.glob('results/hw4/metrics/*.json')):
    with open(path) as f:
        data = json.load(f)
    print(f"\n{'='*50}")
    print(f"  {os.path.basename(path)}")
    print(f"{'='*50}")
    model = data.get('model', '?')
    best  = data.get('best_epoch', '?')
    print(f"  model      : {model}")
    print(f"  best_epoch : {best}")
    if 'mean_mse' in data:
        print(f"  mean_mse   : {data['mean_mse']:.6f}")
        mse_str = ', '.join(f'd{i+1}={v:.4f}' for i, v in enumerate(data['per_horizon_mse']))
        print(f"  per horizon: {mse_str}")
    if 'f1' in data:
        print(f"  accuracy   : {data['accuracy']:.4f}")
        print(f"  precision  : {data['precision']:.4f}")
        print(f"  recall     : {data['recall']:.4f}")
        print(f"  f1         : {data['f1']:.4f}")
        cm = data['confusion_matrix']
        print(f"  conf_mat   : TN={cm[0][0]}  FP={cm[0][1]}  FN={cm[1][0]}  TP={cm[1][1]}")

## 8. Sonuçlar — Loss Grafikleri

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

plots = sorted(glob.glob('results/hw4/plots/*.png'))
if not plots:
    print('Henüz plot yok.')
else:
    fig, axes = plt.subplots(1, len(plots), figsize=(6 * len(plots), 4))
    if len(plots) == 1:
        axes = [axes]
    for ax, path in zip(axes, plots):
        ax.imshow(mpimg.imread(path))
        ax.set_title(os.path.basename(path).replace('_loss.png', ''), fontsize=9)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

## 9. Sonuçları İndir

`results/hw4/` altındaki tüm dosyalar (metrikler, plotlar) tek zip olarak indirilir.  
Checkpointler büyük olduğu için dahil edilmez.

In [ ]:
import shutil
from google.colab import files

# Sadece metrics ve plots — checkpoints hariç
tmp_dir = '/tmp/hw4_results'
os.makedirs(tmp_dir, exist_ok=True)
for sub in ('metrics', 'plots'):
    shutil.copytree(f'results/hw4/{sub}', f'{tmp_dir}/{sub}', dirs_exist_ok=True)

zip_path = '/tmp/hw4_results'
shutil.make_archive(zip_path, 'zip', tmp_dir)
print('hw4_results.zip hazır, indiriliyor...')
files.download(zip_path + '.zip')